# core

> YouTube Transcript to structured Table of Contents

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

def fmt_duration(seconds: int # Duration in seconds
                ) -> str: # Formatted as H:MM:SS or M:SS
    "Format seconds as human-readable duration."
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

def format_header(meta: dict # meta.json content
                 ) -> str: # Formatted header string
    "Shared header for toc/sum/raw CLI commands."
    title = meta.get('title', '')
    channel = meta.get('channel', '')
    dur = fmt_duration(meta.get('duration', 0))
    upload = meta.get('upload_date', '')
    return f'# {title}\nChannel: {channel} | Duration: {dur} | {upload}'

def slice_segments(segments: list[dict], # [{start, end, text}, ...]
                   start: int, # Section start in seconds
                   end: int # Section end in seconds
                  ) -> list[dict]: # Segments within [start, end)
    "Return segments that fall within the given time range."
    return [s for s in segments if s['start'] >= start and s['start'] < end]

def format_toc_line(section: dict, # {path, title, start, end}
                    url: str = '' # webpage_url for &t= deep link (omit when empty)
                   ) -> str: # Single-line TOC entry
    "Format a TOC section as 'N. title H:MM:SS-H:MM:SS (span) URL&t=N'."
    s_start = fmt_duration(section['start'])
    s_end = fmt_duration(section['end'])
    span = fmt_duration(section['end'] - section['start'])
    suffix = f" {url}&t={section['start']}" if url else ''
    return f"{section['path']}. {section['title']} {s_start}-{s_end} ({span}){suffix}"

In [ ]:
# Test: fmt_duration
assert fmt_duration(3991) == '1:06:31'
assert fmt_duration(195) == '3:15'
assert fmt_duration(0) == '0:00'
assert fmt_duration(60) == '1:00'
assert fmt_duration(3600) == '1:00:00'
print('ok')

In [ ]:
# Test: format_header
hdr = format_header({
    'title': 'Test Video', 'channel': 'Test Channel',
    'duration': 3991, 'upload_date': '20260320'})
assert hdr == '# Test Video\nChannel: Test Channel | Duration: 1:06:31 | 20260320'

# Missing fields default to empty
hdr2 = format_header({'title': 'X'})
assert '# X' in hdr2
assert 'Duration: 0:00' in hdr2
print('ok')

In [ ]:
# Test: format_toc_line
sec = {'path': '1', 'title': 'Intro', 'start': 0, 'end': 137}
url = 'https://www.youtube.com/watch?v=ABC'
assert format_toc_line(sec, url) == '1. Intro 0:00-2:17 (2:17) https://www.youtube.com/watch?v=ABC&t=0'

sec2 = {'path': '3', 'title': 'Deep dive', 'start': 600, 'end': 4191}
assert format_toc_line(sec2, url) == '3. Deep dive 10:00-1:09:51 (59:51) https://www.youtube.com/watch?v=ABC&t=600'

# Empty URL: no trailing space, no &t=
assert format_toc_line(sec) == '1. Intro 0:00-2:17 (2:17)'
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()